# Online Learning Platform

In this assignment, you will use the `online_learning_platform.csv` dataset to predict whether a user has renewed their subscription.    
***Notes:***

- Some parts of the code are already provided. **Do not modify the existing code.**
- Write your solution only in the sections marked with `### YOUR SOLUTION`.
- You can verify automatically graded tasks using the cell labeled `### TEST` after each function.

***IMPORTANT NOTE:***
- Name your Jupyter Notebook as `online_learning_platform_{index}.ipynb`.
- For example, if your index is 123456, you should name your notebook as `online_learning_platform_12346.ipynb`.

In [1]:
import os
import hashlib
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.preprocessing import (
    PolynomialFeatures,
    StandardScaler,
    MinMaxScaler,
    LabelEncoder,
    OrdinalEncoder,
)
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV, KFold
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)
from sklearn.linear_model import (
    LinearRegression,
    Lasso,
    Ridge,
    LassoCV,
    RidgeCV,
    LogisticRegression,
)
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor

In [2]:
RANDOM_STATE=42

In [3]:
def hash_data_frame(df):
    df_sorted = df.sort_index(axis=1).sort_values(by=list(df.columns))
    return hashlib.sha256(pd.util.hash_pandas_object(df_sorted, index=True).values).hexdigest()

In [4]:
def hash_series(series):
    series_str = ",".join(map(str, series.values))
    return hashlib.sha256(series_str.encode()).hexdigest()

In [5]:
def check_signature(expected, actual):
    # print(actual)
    try:
        assert actual == expected
    except AssertionError:
        raise

In [6]:
def test_func(func, signature):
    df = pd.read_csv("online_learning_platform.csv")
    df = func(df)
    check_signature(signature, hash_data_frame(df))

In [7]:
def test_partition(func, train_X_signature, test_X_signature, train_y_signature, test_y_signature):
    df = pd.read_csv("online_learning_platform.csv")
    train_X, test_X, train_y, test_y = func(df)
    try:
        # print(hash_data_frame(train_X))
        # print(hash_data_frame(test_X))
        # print(hash_series(train_y))
        # print(hash_series(test_y))
        assert hash_data_frame(train_X) == train_X_signature
        assert hash_data_frame(test_X) == test_X_signature
        assert hash_series(train_y) == train_y_signature
        assert hash_series(test_y) == test_y_signature
    except AssertionError:
        raise

In [8]:
df = pd.read_csv("online_learning_platform.csv")

df.sample()

,student_id,city,age,gender,platform_usage_months,subscription_plan,monthly_logins,avg_study_session_minutes,monthly_study_minutes,engagement_level,subscription_renewed
193,STUDENT-000193,Ohrid,18.0,Female,15.0,Annually,NaN,43.0,223.0,High,1


In [9]:
### AUTOMATICALLY GRADED TASK
def calculate_descriptive_statistics(df):
    """
    Calculate the descriptive statistics for all numeric variables in the dataset.
    The statistics should include: count, mean, standard deviation, minimum,
    25th percentile, median, 75th percentile, and maximum.
    
    Return the result as a `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    return df.describe()
    ### END SOLUTION

In [10]:
statistics = calculate_descriptive_statistics(df)

In [11]:
### TEST
assert isinstance(statistics, pd.DataFrame)
required_stats = ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]
for stat in required_stats:
    assert stat in statistics.index
numeric_columns = df.select_dtypes(include="number").columns
assert list(statistics.columns) == list(numeric_columns)

In [12]:
### AUTOMATICALLY GRADED TASK
def encode_or_drop_student_id(df):
    """
    Encode the `student_id` variable or remove it from the dataset.

    Note: If you plan to perform one-hot encoding, use the `pd.get_dummies` function, 
    use the original column name as a prefix for the new columns and
    append the new columns to the dataset. Also, remove the original column.
    
    Return the dataset as `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    return df.drop(columns=["student_id"])    
    ### END SOLUTION

In [13]:
df = encode_or_drop_student_id(df)

In [14]:
### TEST
test_func(encode_or_drop_student_id, "825c0917fd3bc144ebb2291062ba191f8b2b766a0179bf975aff4a9f9c490335")

In [15]:
### AUTOMATICALLY GRADED TASK
def encode_or_drop_city(df):
    """
    Encode the `city` variable or remove it from the dataset.

    Note: If you plan to perform one-hot encoding, use the `pd.get_dummies` function, 
    use the original column name as a prefix for the new columns and
    append the new columns to the dataset. Also, remove the original column.
        
    Return the dataset as `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    return df.drop(columns=["city"])    
    ### END SOLUTION

In [16]:
df = encode_or_drop_city(df)

In [17]:
### TEST
test_func(encode_or_drop_city, "dc615d9028553f8c938225db961ec9cb07ed9ae9392c13956c0f2238fe36d6d0")

In [18]:
### AUTOMATICALLY GRADED TASK
def encode_or_drop_gender(df):
    """
    Encode the `gender` variable or remove it from the dataset.

    Note: If you plan to perform one-hot encoding, use the `pd.get_dummies` function, 
    use the original column name as a prefix for the new columns and
    append the new columns to the dataset. Also, remove the original column.
    
    Return the dataset as `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    gender_dummies = pd.get_dummies(df["gender"], prefix="gender")
    df = pd.concat([df.drop(columns=["gender"]), gender_dummies], axis=1)
    return df
    ### END SOLUTION

In [19]:
df = encode_or_drop_gender(df)

In [20]:
### TEST
test_func(encode_or_drop_gender, "e6cfadf28186c74e997b01e3f38a55bd4682df8815f3d316b0da7f1d0592d65d")

In [21]:
### AUTOMATICALLY GRADED TASK
def encode_or_drop_subscription_plan(df):
    """
    Encode the `subscription_plan` variable or remove it from the dataset.

    Note: If you plan to perform one-hot encoding, use the `pd.get_dummies` function, 
    use the original column name as a prefix for the new columns and
    append the new columns to the dataset. Also, remove the original column.

    If you encode `subscription_plan`, make sure the encoding reflects the exact duration in months as floats.
    
    Return the dataset as `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    mapping = {
        "Monthly": 1.0,
        "Quarterly": 3.0,
        "Annually": 12.0
    }

    df["subscription_plan"] = df["subscription_plan"].map(mapping)
    return df
    ### END SOLUTION

In [22]:
df = encode_or_drop_subscription_plan(df)

In [23]:
### TEST
test_func(encode_or_drop_subscription_plan, "c2186849a7cc68676da1bfeec5fc565f6c51f440347ec8abfa3cc3af7c2feeed")

In [24]:
### AUTOMATICALLY GRADED TASK
def encode_or_drop_engagement_level(df):
    """
    Encode the `engagement_level` variable or remove it from the dataset.

    Note: If you plan to perform one-hot encoding, use the `pd.get_dummies` function, 
    use the original column name as a prefix for the new columns and
    append the new columns to the dataset. Also, remove the original column.
    
    Return the dataset as `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    encoder = OrdinalEncoder(
        categories=[["Low", "Moderate", "High", "Very High", np.nan]]
    )
    df[["engagement_level"]] = encoder.fit_transform(df[["engagement_level"]])
    return df
    ### END SOLUTION

In [25]:
df = encode_or_drop_engagement_level(df)

In [26]:
### TEST
test_func(encode_or_drop_engagement_level, "8750a37aeaf7e7fb6b152f96de3e1664ce33410da849b286ec48a4c6a6f7ecba")

In [27]:
### AUTOMATICALLY GRADED TASK
def handle_missing_values_in_age(df):
    """
    Impute or remove the missing values from the `age` column.

    Use `random_state=RANDOM_STATE` to ensure reproducibility.

    Return the dataset as a `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    df['age'] = df['age'].fillna(df['age'].median())
    return df
    ### END SOLUTION

In [28]:
df = handle_missing_values_in_age(df)

In [29]:
### TEST
test_func(handle_missing_values_in_age, "20f40111105c437a33e187b2e7114757dd183f66906928732ca748100f21121c")

In [30]:
### AUTOMATICALLY GRADED TASK
def handle_missing_values_in_platform_usage_months(df):
    """
    Impute or remove the missing values from the `platform_usage_months` column.

    Use `random_state=RANDOM_STATE` to ensure reproducibility.

    Return the dataset as a `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    df['platform_usage_months'] = df['platform_usage_months'].fillna(df['platform_usage_months'].median())
    return df
    ### END SOLUTION

In [31]:
df = handle_missing_values_in_platform_usage_months(df)

In [32]:
### TEST
test_func(handle_missing_values_in_platform_usage_months, "9770bf7dee3870fe383e16d009d6bbce748aa9475a5a85b48a0182a3a21356c7")

In [33]:
### AUTOMATICALLY GRADED TASK
def handle_missing_values_in_monthly_logins_and_monthly_study_minutes(df):
    """
    Impute or remove the missing values from the `monthly_logins and monthly_study_minutes` columns.

    Use `random_state=RANDOM_STATE` to ensure reproducibility.

    Return the dataset as a `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    imputer = IterativeImputer(random_state=42)

    df[["monthly_logins", "monthly_study_minutes"]] = imputer.fit_transform(
        df[["monthly_logins", "monthly_study_minutes"]]
    )    
    return df
    ### END SOLUTION

In [34]:
df = handle_missing_values_in_monthly_logins_and_monthly_study_minutes(df)

In [35]:
### TEST
test_func(handle_missing_values_in_monthly_logins_and_monthly_study_minutes, "846d7f8c3e5e86fdb7f9e4817d52247157f1ae7ce38b60f9c25fad7cef10654b")

In [36]:
### AUTOMATICALLY GRADED TASK
def handle_missing_values_in_avg_study_session_minutes(df):
    """
    Impute or remove the missing values from the `avg_study_session_minutes` column.

    Use `random_state=RANDOM_STATE` to ensure reproducibility.

    Return the dataset as a `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    df['avg_study_session_minutes'] = df['avg_study_session_minutes'].fillna(df['avg_study_session_minutes'].median())
    return df
    ### END SOLUTION

In [37]:
df = handle_missing_values_in_avg_study_session_minutes(df)

In [38]:
### TEST
test_func(handle_missing_values_in_avg_study_session_minutes, "4fea88197ea2dcec22a65df0275075d2d6f2de2ae688b1e524144f55e07bbca6")

In [39]:
### AUTOMATICALLY GRADED TASK
def handle_missing_values_in_engagement_level(df):
    """
    Impute or remove the missing values from the `engagement_level` column.

    Use `random_state=RANDOM_STATE` to ensure reproducibility.

    Return the dataset as a `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    df["engagement_level"] = df["engagement_level"].fillna(df["engagement_level"].mode()[0])
    return df
    ### END SOLUTION

In [40]:
df = handle_missing_values_in_engagement_level(df)

In [41]:
### TEST
test_func(handle_missing_values_in_engagement_level, "0f43a71ad15c85f8c390ea76483cbf54f5cc437e860673f0924eb0d25133546f")

In [42]:
### AUTOMATICALLY GRADED TASK
def split_dataset_into_train_and_test(df):
    """
    Split the dataset into features `X` and target `y`, where the target is `subscription_renewed`.
    Then, divide `X` and `y` into training and test sets using an 85:15 ratio,
    ensuring that the class distribution of y is preserved in both sets.

    Use `random_state=RANDOM_STATE` to ensure reproducibility.
    
    Return `train_X`, `test_X`, `train_y`, and `test_y`.
    """

    ### BEGIN SOLUTION
    X, y = df.drop(columns=["subscription_renewed"]), df["subscription_renewed"]

    train_X, test_X, train_y, test_y = train_test_split(
        X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y
    )

    return train_X, test_X, train_y, test_y
    ### END SOLUTION

In [43]:
train_X, test_X, train_y, test_y = split_dataset_into_train_and_test(df)

In [44]:
### TEST
test_partition(
    split_dataset_into_train_and_test,
    "99abee275722cf032e78e85c168a49aae1af8968b354fa7ce395daec031b61c3",
    "1674c77a471ba8c47660823c459f6e94161ec7374fb4df581f4ab4b9e3ada628",
    "0523e73a4f39bc7ed8fdbccac82a33d5897fed196acec9e3cefccb00f9623576",
    "3fd1c9929d2fa8aa46b4922db0c18a3d34783e7868026ea2d8872ddb713ac4fd",
)

In [45]:
### AUTOMATICALLY GRADED TASK
def fit_model(train_X, train_y):
    """
    Fit a boosting model to predict `y` using `X` with 50 estimators, learning rate 0.05 and a maximum depth of 5.

    Use `random_state=RANDOM_STATE` to ensure reproducibility.

    Return the fitted model.
    """

    ### BEGIN SOLUTION
    return XGBClassifier(    
        n_estimators=50,
        learning_rate=0.05,
        max_depth=5,
        random_state = RANDOM_STATE).fit(train_X, train_y)
    ### END SOLUTION    

In [46]:
model = fit_model(train_X, train_y)
pred_y = model.predict(test_X)

In [47]:
### TEST
assert isinstance(model, XGBClassifier)
params = model.get_params()
assert params["n_estimators"] == 50
assert params["max_depth"] == 5
assert abs(params["learning_rate"] - 0.05) < 1e-12
assert params["random_state"] == RANDOM_STATE
assert pred_y.shape[0] == test_X.shape[0]

In [48]:
def evaluate_model(test_y, pred_y):
    """
    Evaluate the model using precision, recall, and F1-score, with a weighted average.

    Return `precision`, `recall`, and `f1` rounded with 2 decimals.
    """
    
    ### BEGIN SOLUTION
    precision = precision_score(test_y, pred_y, average="weighted")
    recall = recall_score(test_y, pred_y, average="weighted")
    f1 = f1_score(test_y, pred_y, average="weighted")

    return round(precision, 2), round(recall, 2), round(f1, 2)
    ### END SOLUTION


In [49]:
precision, recall, f1 = evaluate_model(test_y, pred_y)

In [50]:
### TEST
assert precision > 0.70
assert recall > 0.70
assert f1 > 0.70